In [1]:
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("train-labels.csv")

df = df[df["text"].str.len() == 6].copy()

print(df.shape)

(19998, 3)


In [3]:
characters = sorted(set("".join(df["text"])))

char_to_num = {
    char: idx
    for idx, char in enumerate(characters)
}

num_to_char = {
    idx: char
    for idx, char in enumerate(characters)
}

print("Vocabulary Size:", len(characters))

Vocabulary Size: 31


In [4]:
def preprocess_image(path):

    img = cv2.imread(
        path,
        cv2.IMREAD_GRAYSCALE
    )

    img = img.astype(np.float32)

    img = img/255.0

    img = np.expand_dims(
        img,
        axis=-1
    )

    return img

In [5]:
def encode_label(text):

    return np.array(
        [char_to_num[c] for c in text],
        dtype=np.int32
    )

In [6]:
image_paths = (
    "train_images/" +
    df["image"]
).values

labels = df["text"].values

In [7]:
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.1,
    random_state=42)  #90% training 10% validation

In [8]:
print("Training:", len(train_paths))
print("Validation:", len(val_paths))

Training: 17998
Validation: 2000


In [9]:
def load_sample(path,label):

    path = path.numpy().decode()

    label = label.numpy().decode()

    image = preprocess_image(path)

    label = encode_label(label)

    return image, label

In [10]:
def tf_load_sample(path, label):

    image, label = tf.py_function(
        load_sample,
        [path, label],
        [tf.float32, tf.int32]
    )

    image.set_shape((100, 200, 1))

    label.set_shape((6,))

    return image, label

In [11]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_paths, val_labels)
)

In [12]:
train_ds = train_ds.map(
    tf_load_sample,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds = val_ds.map(
    tf_load_sample,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [13]:
train_ds = train_ds.shuffle(
    1000
)

In [14]:
BATCH_SIZE = 32

train_ds = train_ds.batch(
    BATCH_SIZE
)

val_ds = val_ds.batch(
    BATCH_SIZE
)

In [15]:
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

val_ds = val_ds.prefetch(
    tf.data.AUTOTUNE
)

In [16]:
for images, labels in train_ds.take(1):

    print(images.shape)
    print(labels.shape)

(32, 100, 200, 1)
(32, 6)
